# BusBooking · Pha 2B trên Kaggle CPU

Chọn **Accelerator: None** và **Internet: On**. Gắn dataset riêng chứa các tệp trong bảng bên dưới rồi điền `B_SHA` ở cell kế tiếp. Chạy toàn bộ notebook trong cùng một phiên Kaggle.

| Tệp | Nội dung |
|---|---|
| `busbooking-p1.jar` | JAR chạy được từ commit `fd6b5f3026412d2359d70bfad681fc0773c8bcae` |
| `busbooking-b.jar` | JAR chạy được từ commit triển khai B |
| `benchmark.py`, `locustfile.py`, `fixture.sql` | Đặt cùng thư mục trong dataset |

Mỗi bản chạy 3 lượt cho từng workload. Cả hai workload đều gửi `Idempotency-Key`. Workload retry gửi 100 cặp tuần tự và 100 cặp đồng thời trên 1.000 ý định. Mỗi lượt có database khôi phục từ cùng snapshot Pha 1 và một lượt warm-up riêng.

In [ ]:
import csv, os, pwd, re, secrets, shutil, subprocess, sys, tempfile
from datetime import datetime, timezone
from pathlib import Path

B_SHA = ''  # SHA đầy đủ của commit dùng để build busbooking-b.jar
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/busbooking-phase2-b')
WORK.mkdir(exist_ok=True)

def find_input(name):
    files = sorted(INPUT.rglob(name))
    if len(files) != 1:
        raise ValueError(f'Cần đúng một tệp {name}, tìm thấy {files}')
    return files[0]

if not re.fullmatch(r'[a-f0-9]{40}', B_SHA):
    raise ValueError('Điền SHA đầy đủ của bản B vào B_SHA.')
p1_jar = find_input('busbooking-p1.jar')
b_jar = find_input('busbooking-b.jar')
benchmark = find_input('benchmark.py')
for name in ('locustfile.py', 'fixture.sql'):
    if not (benchmark.parent / name).is_file():
        raise ValueError(f'Đặt {name} cùng thư mục với benchmark.py')
print('Đã tìm thấy hai JAR và bộ benchmark.')

In [ ]:
def run(command, **kwargs):
    return subprocess.run([str(part) for part in command], check=True, text=True, **kwargs)

run([sys.executable, '-m', 'pip', 'install', '-q', 'locust==2.32.10', 'psutil==6.1.1',
     'requests==2.32.3', 'psycopg[binary]==3.2.4'])

java = shutil.which('java')
if not java or not re.search(r'version "21(?:[."]|$)', run([java, '-version'], capture_output=True).stderr):
    java_dir = WORK / 'jdk21'
    java = java_dir / 'bin/java'
    if not java.is_file():
        java_dir.mkdir(exist_ok=True)
        java_archive = WORK / 'jdk21.tar.gz'
        run(['curl', '-fL', '--retry', '3', '-o', java_archive,
             'https://corretto.aws/downloads/latest/amazon-corretto-21-x64-linux-jdk.tar.gz'])
        run(['tar', '-xzf', java_archive, '--strip-components=1', '-C', java_dir])
        java_archive.unlink()
run([java, '-version'])

def postgres_versions():
    return [path for path in Path('/usr/lib/postgresql').glob('[0-9]*/bin/initdb')
            if int(path.parent.parent.name) >= 14 and Path('/usr/share/postgresql',
                path.parent.parent.name, 'extension', 'pg_stat_statements.control').is_file()]

if not postgres_versions():
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', '-qq', 'postgresql', 'postgresql-contrib'],
        env={**os.environ, 'DEBIAN_FRONTEND': 'noninteractive'})
pg_bin = max(postgres_versions(), key=lambda path: int(path.parent.parent.name)).parent
print(f'PostgreSQL executables: {pg_bin}')

In [ ]:
pg_user = pwd.getpwnam('postgres')
pg_root = Path(tempfile.mkdtemp(prefix='busbooking-phase2-b-'))
os.chown(pg_root, pg_user.pw_uid, pg_user.pw_gid)
pg_data = pg_root / 'data'
pg_port = 55433
password = secrets.token_urlsafe(30)
password_file = pg_root / 'password'
password_file.write_text(password + '\n', encoding='ascii')
password_file.chmod(0o600)
os.chown(password_file, pg_user.pw_uid, pg_user.pw_gid)
as_postgres = ['runuser', '-u', 'postgres', '--']
try:
    run([*as_postgres, pg_bin / 'initdb', '-D', pg_data, '-U', 'postgres', '--encoding=UTF8',
         '--locale=C.UTF-8', '--auth-local=peer', '--auth-host=scram-sha-256', f'--pwfile={password_file}'])
finally:
    password_file.unlink(missing_ok=True)
run([*as_postgres, pg_bin / 'pg_ctl', '-D', pg_data, '-l', pg_root / 'postgresql.log', '-w', 'start',
     '-o', f'-p {pg_port} -h 127.0.0.1 -c shared_preload_libraries=pg_stat_statements'])

run_id = datetime.now(timezone.utc).strftime('phase2-b-%Y%m%dT%H%M%SZ')
output = WORK / run_id
try:
    pg_pid = (pg_data / 'postmaster.pid').read_text().splitlines()[0]
    run([sys.executable, '-u', benchmark, '--java', java, '--pg-port', pg_port, '--pg-pid', pg_pid,
         '--p1-jar', p1_jar, '--b-jar', b_jar, '--b-sha', B_SHA, '--output', output],
        env={**os.environ, 'PHASE2_PG_PASSWORD': password})
finally:
    try:
        run([*as_postgres, pg_bin / 'pg_ctl', '-D', pg_data, '-m', 'fast', '-w', 'stop'])
    finally:
        if output.is_dir():
            shutil.copyfile(pg_root / 'postgresql.log', output / 'postgresql.log')
            zip_path = shutil.make_archive(str(output), 'zip', root_dir=output.parent, base_dir=output.name)
            print(f'Tải ZIP trong Kaggle Output: {zip_path}')

In [ ]:
from IPython.display import FileLink, Markdown, display

with (output / 'summary.csv').open(encoding='utf-8', newline='') as stream:
    results = list(csv.DictReader(stream))

columns = {
    'repetition': 'Lượt', 'variant': 'Bản', 'workload': 'Workload', 'requests': 'Request',
    'bookings': 'Booking', 'duplicate_bookings': 'Booking thừa', 'extra_seats': 'Ghế thừa',
    'first_attempt_p95_ms': 'p95 lần đầu (ms)', 'retry_attempt_p95_ms': 'p95 gửi lại (ms)',
    'replayed_200_p95_ms': 'p95 HTTP 200 (ms)', 'checked': 'Đối soát',
}
table = [' | '.join(columns.values()), ' | '.join(['---'] * len(columns))]
table += [' | '.join(row[key] or '—' for key in columns) for row in results]
display(Markdown('\n'.join(f'| {line} |' for line in table)))
display(FileLink(zip_path))